Step 1: Install, bootstrap instruction

In [ ]:
!pip install --quiet anthropic pydantic
!pip install --upgrade --quiet ipython

from google.colab import userdata, drive
drive.mount('/content/drive')

import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
os.environ["ASTRA_CASSETTE_DIR"] = "/content/drive/MyDrive/astra-swarm/cassettes"

!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys
sys.path.insert(0, "/content/astra-swarm/src")

%load_ext autoreload
%autoreload 2

# Health check
import astra_swarm.alerts, astra_swarm.schemas, astra_swarm.agent_loop, astra_swarm.tools
from astra_swarm.cassette import cassette
import subprocess
sha = subprocess.check_output(
    ["git", "-C", "/content/astra-swarm", "rev-parse", "--short", "HEAD"]
).decode().strip()
print(f"repo @ {sha} — ready")

Step 2: Generate 10 alerts

In [ ]:
import json
from pathlib import Path
from astra_swarm.alerts import generate_synthetic_alerts

alerts = []
with cassette("day05_synthetic_alerts", mode="auto"):
    alerts = generate_synthetic_alerts(10)

fixture_path = Path("/content/astra-swarm/data/synthetic/10_alerts.json")
fixture_path.parent.mkdir(parents=True, exist_ok=True)
fixture_path.write_text(json.dumps(alerts, indent=2))

for i, a in enumerate(alerts, 1):
    print(f"--- Alert {i} ---\n{a}\n")

Step 3: Run full chain, cassette-backed

In [ ]:
from astra_swarm.alerts import triage_chain
from astra_swarm.cassette import cassette

results = []
with cassette("week01_milestone", mode="auto"):
    for i, alert in enumerate(alerts, 1):
        print(f"[{i}/{len(alerts)}] triaging...", end=" ", flush=True)
        try:
            r = triage_chain(alert)
            results.append(r)
            print(f"OK  severity={r.verdict.severity.value}  "
                  f"techs={len(r.attack.techniques)}  conf={r.verdict.confidence:.2f}")
        except Exception as e:
            print(f"FAIL: {type(e).__name__}: {e}")
            results.append(None)   # keep positional alignment with `alerts`

# Persist for the retrospective and future comparison
out = Path("/content/astra-swarm/data/synthetic/week01_triage_results.json")
out.write_text(json.dumps(
    [r.model_dump() if r else {"error": "chain_failed"} for r in results],
    indent=2, default=str,
))
print(f"\nwrote {out}")

Step 4: Compute basic metrics

In [ ]:
import statistics
from collections import Counter

valid = [r for r in results if r is not None]

# --- Severity distribution ---
sev_dist = Counter(r.verdict.severity.value for r in valid)

# --- ATT&CK techniques cited ---
all_techs = [t.id for r in valid for t in r.attack.techniques]
tech_freq = Counter(all_techs)

# --- Graceful degradation rate (empty techniques via max_rounds path) ---
degraded = sum(
    1 for r in valid
    if not r.attack.techniques and "inconclusive" in r.attack.summary.lower()
)

# --- Confidence distribution ---
confs = [r.verdict.confidence for r in valid]

# --- Alerts with NO ATT&CK match (honest "no fit" from the model) ---
no_technique_honest = sum(
    1 for r in valid
    if not r.attack.techniques and "inconclusive" not in r.attack.summary.lower()
)

print("=" * 60)
print(f"Week-1 milestone: {len(valid)}/{len(alerts)} alerts triaged")
print("=" * 60)
print(f"\nSeverity distribution:")
for sev, n in sev_dist.most_common():
    print(f"  {sev:<10} {'█' * n}  {n}")
print(f"\nATT&CK techniques cited: {len(all_techs)} total, {len(tech_freq)} unique")
print(f"Top techniques:")
for tid, n in tech_freq.most_common(5):
    print(f"  {tid:<10} × {n}")
print(f"\nGraceful degradation (max_rounds hit): {degraded}/{len(valid)}")
print(f"Honest 'no clear ATT&CK fit':          {no_technique_honest}/{len(valid)}")
print(f"\nConfidence: mean={statistics.mean(confs):.2f}, "
      f"min={min(confs):.2f}, max={max(confs):.2f}")